In [1]:
import pandas as pd
from mopac_batch_runner import prepare_batch, run_mopac_batch, export_sdf

In [3]:
df = pd.read_excel('./datasets/ionic_structures_test.xlsx')


df_100 = df.head(100)

In [4]:
df_100.head()

,compound_id,smiles
0,1,[O-][N+](=O)C1=CC=C(Cl)C(Cl)=C1Cl
1,2,[O-][N+](=O)C1=C(Cl)C=C(Cl)C=C1Cl
2,3,OCC(CO)(CO)[N+]([O-])=O
3,4,OCCNC1=CC=C(C=C1[N+]([O-])=O)N(CCO)CCO
4,5,OCCNC1=CC=C(C=C1OCCO)[N+]([O-])=O


In [5]:
# Step 1: SMILES → 3D structures → .mop files
prep = prepare_batch(
    df=df_100,
    smiles_col="smiles",
    id_col="compound_id",
    keywords="PM7 GEO-OK",
    workdir="mopac_jobs",
    auto_charge=True,   # ← detects [NH4+], [O-], etc. automatically
    overwrite=True,
    max_workers=6,
)

In [6]:
# verify if anything failed
print(prep[prep['status'] != 'ready'])

Empty DataFrame
Columns: [compound_id, smiles, status, mop_file, error_message, duration_s]
Index: []


In [7]:
# Step 2: run MOPAC on all prepared .mop files
results = run_mopac_batch(
    workdir="mopac_jobs",
    overwrite=True,
    timeout=900,
    max_workers=6,
)



In [ ]:
print(results[results['status'] != 'success'])

Empty DataFrame
Columns: [compound_id, smiles, status, mop_file, out_file, arc_file, error_message, return_code, duration_s]
Index: []


In [9]:
print(results[results['arc_file'].isna()])#.to_csv('failed_runs.csv')

Empty DataFrame
Columns: [compound_id, smiles, status, mop_file, out_file, arc_file, error_message, return_code, duration_s]
Index: []


In [10]:
results

,compound_id,smiles,status,mop_file,out_file,arc_file,error_message,return_code,duration_s
0,1,[O-][N+](=O)C1=CC=C(Cl)C(Cl)=C1Cl,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,0.954
1,10,[O-][N+](=O)C1=CC(Cl)=C(Cl)C(Cl)=C1,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,0.492
2,11,CN=C(NCC1CCOC1)N[N+]([O-])=O,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,1.499
3,12,[O-][N+](=O)C1=CC=C2C(=O)OC(=O)C2=C1,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,0.475
4,13,OC1=CC=C(C2=C1C(=O)C1=C(C2=O)C(=CC=C1O)[N+]([O...,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,15.441
...,...,...,...,...,...,...,...,...,...
72,75,CCCCC1C(=O)N(N(C1=O)C1=CC=CC=C1)C1=CC=CC=C1,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,5.736
73,76,CC12CC(=O)C3C(CCC4CC(O)CCC34C)C1CCC2(O)C(O)CO,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,5.410
74,77,ClC1=C(Cl)C2(Cl)C3C4CC(C5OC45)C3C1(Cl)C2(Cl)Cl,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,2.486
75,8,[O-][N+](=O)C(Br)Br,success,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,C:\Users\macie\Documents\GitHub\smi_to_mop\mop...,None,0,0.175


In [14]:
# Step 3: export optimised geometries to SDF
sdf = export_sdf(
    workdir="mopac_jobs",
    overwrite=True,
    max_workers=6,
)



In [13]:
print(sdf[sdf['status'] != 'success'])

Empty DataFrame
Columns: [compound_id, arc_file, sdf_file, status, error_message, duration_s]
Index: []
